In [ ]:
# Cell 1 — configure HPC sources and a NEW diagnostic output directory.
from pathlib import Path
import hashlib,json,platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
V12=Path('/data1/huangy/fig6/NC/v12/four_init_full_pipeline_NCVIconcurrent')
NCVI_RUN=V12/'ncvi_daily_standalone/run_001'
RAW_DAILY=V12/'tables/table_daily_member_raw_factors_SHF.csv'
TCC_DAILY=V12/'tables/table_daily_member_tcc_SHF.csv'
TMAX_DAILY=V12/'tables/table_daily_tmax_SHF.csv'
STAGE=V12/'tables/table_stage_member_raw_factors_TCC.csv'
MODEL=V12/'mlr_raw_residual_comparison'
# Existing reference table for other ERA5 physical fields (does not contain validated NCVI).
# Set ERA5_PHYSICAL=None only to omit those OTHER ERA5 reference lines.
ERA5_PHYSICAL=Path('/data1/huangy/fig6/NC/v11/hotcold_stage_group_timeseries_tables_multiinit_totalfig6/tables/daily_timeseries_all_factors.csv')
# Independent ERA5 theta=320 K NCVI daily regional-mean anomaly, already computed.
# Mandatory input for Fig.5/11 NCVI diagnostic; do NOT rerun ERA5 or change S2S.
ERA5_NCVI_CSV=Path('/data1/huangy/fig6/NC/MSE/ERA5/NCVI_theta320/processed/era5_ncvi_theta320_daily_anomaly_2023.csv')
# Actual HPC CSV header confirmed from the Cell3 traceback on 2026-10-14.
# Use exact columns rather than accidentally loading abs/clim PV; None remains
# supported for strict unambiguous column autodetection when changing sources.
ERA5_NCVI_DATE_COL='date'
ERA5_NCVI_VALUE_COL='ncvi_theta320_anom_pvu'
# Source unit expectation; an actual CSV units column takes precedence and is checked.
# Without independent metadata, units/climatology/region remain PROVENANCE_PENDING.
ERA5_NCVI_EXPECTED_UNIT='PVU'
OUTPUT=V12/'process_plot_tables/run_001'
if OUTPUT.exists():raise ValueError('Choose a NEW output directory')
if OUTPUT.resolve()==NCVI_RUN.resolve() or NCVI_RUN.resolve() in OUTPUT.resolve().parents:raise ValueError('Output must be outside immutable NCVI run')
INITS=('2023-06-08','2023-06-12');GROUPS=('Hot17','Middle17','Cold17')
VERSIONS=('residualized','raw')
FACTORS5=('NCHN_tp','z500_anom_NCHN','sm_avg','NCVI_concurrent','TCC_Avg')
FACTORS11=('sm_avg','z500_anom_NCHN','SHF_Avg','NCVI_concurrent','TCC_Avg')
PREDICTORS={'residualized':('sm_avg','SHF_resid','z500_anom_NCHN','Initial_MSEstar_max_NCHN','TCC_resid','NCVI_concurrent'),
'raw':('sm_avg','SHF_Avg','z500_anom_NCHN','Initial_MSEstar_max_NCHN','TCC_Avg','NCVI_concurrent')}
FACTORS=('y_tmax','NCHN_tp','sm_avg','SHF_Avg','z500_anom_NCHN','TCC_Avg');KEY=['init_date','date','member']
SNAPSHOTS={}
def capture(number):
 text=get_ipython().history_manager.input_hist_raw[-1]
 if not text.strip():raise RuntimeError('Missing execution snapshot')
 SNAPSHOTS[number]=text
 for k in list(SNAPSHOTS):
  if k>number:del SNAPSHOTS[k]
capture(1)
print('Cell 1 READY; independent ERA5 NCVI path configured; source provenance remains PENDING')


In [ ]:
# Cell 2 — independent physical-source and saved-model verification.
capture(2)
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def require(df, names, context):
    missing = set(names) - set(df.columns)
    if missing:
        raise ValueError(f'{context}: missing fields {sorted(missing)}')

def finite(values, context):
    a = np.asarray(values, float)
    if not np.isfinite(a).all():
        raise ValueError(f'{context}: missing or nonfinite values; no rows dropped')
    return a

def normalize_window(series):
    return series.replace({'Stage-I_dry': 'Stage-I', 'Stage_I': 'Stage-I'})

def parameters(data, init, version):
    """Load a complete saved Stage-I model and validate its 51-member design."""
    summary, coef, design = data['summary'], data['coefficients'], data['design']
    require(summary, ['init_date', 'window', 'version', 'n', 'design_rank', 'predictor_list', 'standardized_intercept'], 'model summary')
    require(coef, ['init_date', 'window', 'version', 'predictor', 'position', 'standardized_coef', 'mean', 'scale_ddof0'], 'coefficients')
    require(design, ['init_date', 'window', 'version', 'member', 'predictor', 'value', 'y_tmax'], 'training design')
    select = lambda d: d[(d.init_date == init) & (d.window == 'Stage-I') & (d.version == version)].copy()
    s = select(summary)
    c = select(coef)
    x = select(design)
    if len(s) != 1 or s.iloc[0]['n'] != 51 or s.iloc[0].design_rank != 7 or s.iloc[0].predictor_list != ';'.join(PREDICTORS[version]):
        raise ValueError(f'{init}/{version}: invalid complete final Stage-I model')
    c = c[c.predictor != 'const'].sort_values('position')
    if c.predictor.tolist() != list(PREDICTORS[version]) or c.position.tolist() != list(range(1, 7)):
        raise ValueError(f'{init}/{version}: coefficients are not the final six predictors')
    finite(c[['standardized_coef', 'mean', 'scale_ddof0']], 'saved scaler/coefficients')
    if (c.scale_ddof0 <= 0).any():
        raise ValueError('Saved scaler must have positive scales')
    if len(x) != 306 or x.duplicated(['member', 'predictor']).any():
        raise ValueError(f'{init}/{version}: incomplete training design')
    for row in c.itertuples():
        q = x[x.predictor == row.predictor].sort_values('member')
        if not np.array_equal(q.member.to_numpy(), np.arange(51)):
            raise ValueError('Training design member coverage mismatch')
        a = finite(q.value, 'training predictors')
        if not np.isclose(a.mean(), row.mean, rtol=0, atol=1e-8) or not np.isclose(a.std(ddof=0), row.scale_ddof0, rtol=0, atol=1e-8):
            raise ValueError('Training design does not reproduce saved scaler')
    pivot = x.pivot(index='member', columns='predictor', values='value').loc[:, list(PREDICTORS[version])]
    y = x.pivot(index='member', columns='predictor', values='y_tmax')
    finite(y, 'training y_tmax')
    if not (y.to_numpy() == y.iloc[:, 0].to_numpy()[:, None]).all():
        raise ValueError('Training response differs by predictor position')
    intercept = float(s.iloc[0].standardized_intercept)
    finite(intercept, 'standardized intercept')
    fitted = select(data['fitted']).sort_values('member')
    require(fitted, ['member','y_tmax','y_fitted','residual'], 'saved fitted members')
    if len(fitted) != 51 or not np.array_equal(fitted.member, np.arange(51)):
        raise ValueError('Saved fitted members incomplete')
    reconstructed = intercept + ((pivot.to_numpy() - c['mean'].to_numpy()) /
                                  c.scale_ddof0.to_numpy()) @ c.standardized_coef.to_numpy()
    finite(fitted[['y_tmax','y_fitted','residual']], 'saved fitted members')
    if not np.allclose(reconstructed, fitted.y_fitted, atol=1e-6, rtol=0) or not np.allclose(y.iloc[:,0], fitted.y_tmax, atol=1e-8, rtol=0):
        raise ValueError('Full-model replay differs from saved fit/Stage-I training response; coefficients are not replaced')
    if not np.allclose(fitted.y_tmax-fitted.y_fitted, fitted.residual, atol=1e-8, rtol=0):
        raise ValueError('Saved fitted residual identity failed')
    mean_state_prediction = float(intercept + ((pivot.mean().to_numpy()-c['mean'].to_numpy()) / c.scale_ddof0.to_numpy()) @ c.standardized_coef.to_numpy())
    if not np.isclose(mean_state_prediction, intercept, atol=1e-8, rtol=0) or not np.isclose(mean_state_prediction, fitted.y_fitted.mean(), atol=1e-8, rtol=0):
        raise ValueError('Diagnostic baseline does not equal full-model prediction at training mean state')
    return {'intercept': intercept, 'coef': c.set_index('predictor'),
            'design': pivot, 'y': y.iloc[:, 0].sort_index(), 'version': version,
            'mean_state_prediction': mean_state_prediction,
            'full_fit_max_abs_difference': float(np.max(np.abs(reconstructed-fitted.y_fitted.to_numpy())))}

def residual_relation(data, init):
    a = data['residualization']
    require(a, ['init_date', 'window', 'target', 'source', 'control', 'intercept', 'slope'], 'residualization audit')
    a = a[(a.init_date == init) & (a.window == 'Stage-I') & (a.target == 'SHF_resid')]
    if len(a) != 1 or a.iloc[0].source != 'SHF_Avg' or a.iloc[0].control != 'sm_avg':
        raise ValueError(f'{init}: require saved Stage-I SHF_Avg ~ intercept + sm_avg residual relation')
    return tuple(finite(a[['intercept', 'slope']], 'fixed residual relation')[0])

def partial_fitted_tmax(sm_value, shf_value, model, relation):
    """Actual diagnostic entry: alpha_std + beta_SM*z_SM + beta_SHF*z_SHF.

    All other six-factor full-model predictors remain at their own training means.
    Saved stage-member residual parameters are used without any daily fitting.
    """
    smv, shfv = np.asarray(sm_value, float), np.asarray(shf_value, float)
    factor = 'SHF_resid' if model['version'] == 'residualized' else 'SHF_Avg'
    projected = shfv - (relation[0] + relation[1] * smv) if factor == 'SHF_resid' else shfv
    c = model['coef']
    value = (model['intercept'] + c.loc['sm_avg', 'standardized_coef'] *
             (smv - c.loc['sm_avg', 'mean']) / c.loc['sm_avg', 'scale_ddof0'] +
             c.loc[factor, 'standardized_coef'] *
             (projected - c.loc[factor, 'mean']) / c.loc[factor, 'scale_ddof0'])
    return value, projected
def adapt_v12_daily(raw, tcc, tmax, stage):
    """Return physical long data, current groups and daily-to-stage numeric audit.

    Dates retain source labels: TCC UTC interval metadata is preserved and
    explicitly audited separately, rather than silently shifted to BJT days.
    """
    frames = []
    dates = pd.date_range('2023-06-14', '2023-06-24')
    expected = pd.MultiIndex.from_product([INITS, dates, range(51)], names=KEY)
    for name, source in [('raw', raw), ('tcc', tcc), ('tmax', tmax)]:
        d = source[source.init_date.isin(INITS)].copy()
        d['date'] = pd.to_datetime(d.date, errors='raise')
        if d.duplicated(KEY).any() or not pd.MultiIndex.from_frame(d[KEY]).sort_values().equals(expected.sort_values()):
            raise ValueError(f'{name}: require exactly 2 inits x 11 dates x 51 integer members')
        frames.append(d)
    d, c, t = frames
    d = d.merge(c[KEY + ['TCC_Avg']], on=KEY, validate='one_to_one')
    check = d.merge(t[KEY + ['s2s_tmax']], on=KEY, validate='one_to_one')
    if not np.array_equal(check.y_tmax.to_numpy(), check.s2s_tmax.to_numpy()):
        raise ValueError('Daily Tmax tables differ')
    finite(d[list(FACTORS)], 'v12 physical factors')
    s = stage.copy(); s['window'] = normalize_window(s.stage)
    groups, audit, pieces = [], [], []
    for init in INITS:
        for window, end in [('Stage-I', '2023-06-17'), ('Total', '2023-06-24')]:
            target = s[(s.init_date == init) & (s.window == window)].set_index('member').sort_index()
            if len(target) != 51 or not np.array_equal(target.index, np.arange(51)):
                raise ValueError('Stage input member coverage invalid')
            ranking = target.y_tmax.sort_values(ascending=False)
            if ranking.iloc[16] == ranking.iloc[17] or ranking.iloc[33] == ranking.iloc[34]:
                raise ValueError('Boundary tie requires explicit scientific ranking rule')
            g = pd.DataFrame({'member': ranking.index, 'rank': range(1,52)})
            g['group'] = [GROUPS[i//17] for i in range(51)]
            g['init_date'], g['window'] = init, window
            groups.append(g)
            q = d[(d.init_date == init) & (d.date <= end)]
            for factor in FACTORS:
                rebuilt = q.groupby('member')[factor].sum() if factor == 'NCHN_tp' else q.groupby('member')[factor].mean()
                for member in range(51):
                    delta = float(rebuilt.loc[member] - target.loc[member, factor])
                    audit.append(dict(init_date=init, window=window, member=member, factor=factor,
                                      daily_aggregate=rebuilt.loc[member], stage_value=target.loc[member,factor],
                                      difference=delta, aggregation='sum' if factor=='NCHN_tp' else 'mean',
                                      passed=abs(delta)<=1e-8))
            display = d[d.init_date == init].copy()
            display = display.merge(g[['member','group']], on='member', validate='many_to_one')
            display['window'] = window
            long = display.melt(id_vars=KEY+['window','group'], value_vars=list(FACTORS), var_name='factor', value_name='value')
            pieces.append(long)
    qc = pd.DataFrame(audit)
    if not qc.passed.all():
        raise ValueError('v12 daily-to-stage consistency FAILED; do not export')
    return pd.concat(pieces, ignore_index=True), pd.concat(groups, ignore_index=True), qc

def group_daily(long):
    """Group means use current v12 membership, never legacy grouped curves."""
    keys = ['init_date','window','date','group','factor']
    counts = long.groupby(keys).size()
    if not (counts == 17).all():
        raise ValueError('Each physical group requires exactly 17 members')
    return long.groupby(keys, as_index=False).value.mean()

inputs={'raw_daily':RAW_DAILY,'tcc_daily':TCC_DAILY,'tmax_daily':TMAX_DAILY,'stage':STAGE,
'comparison_manifest':MODEL/'qc/comparison_manifest.json'}
for key,name in {'summary':'table_mlr_model_summary.csv','coefficients':'table_mlr_model_coefficients.csv',
'design':'table_mlr_design_members.csv','fitted':'table_mlr_fitted_members.csv','residualization':'table_mlr_residualization_audit.csv'}.items():inputs[key]=MODEL/'tables'/name
hashes={k:{'path':str(p),'sha256':sha256(p)} for k,p in inputs.items()}
mm=json.loads(inputs['comparison_manifest'].read_text())
for key in ['summary','coefficients','design','fitted','residualization']:
 if hashes[key]['sha256']!=mm['table_sha256'][inputs[key].name]:raise ValueError('Model hash mismatch')
if hashes['stage']['sha256']!=next(x['sha256'] for x in mm['provenance'] if x['name']=='raw'):raise ValueError('Stage source hash mismatch')
data={k:pd.read_csv(inputs[k]) for k in ['summary','coefficients','design','fitted','residualization']}
physical_long,current_groups,physical_qc=adapt_v12_daily(pd.read_csv(RAW_DAILY),pd.read_csv(TCC_DAILY),pd.read_csv(TMAX_DAILY),pd.read_csv(STAGE))
physical_group=group_daily(physical_long)
print('Cell 2 PASS; v12 grouping and 1224 daily-to-stage checks; max error',physical_qc.difference.abs().max())


In [ ]:
# Cell 3 — authorize diagnostic use only; preserve original NCVI FAIL.
capture(3)

def load_era5_ncvi_reference(path, date_column=None, value_column=None, expected_unit='PVU'):
    """Strictly inspect a precomputed ERA5 320-K NCVI regional anomaly CSV.

    Never infer units from numerical magnitude or turn absolute PV into anomaly.
    CSV schema is not assumed; ambiguous or unknown headers fail with the
    candidate list so Cell1 can be configured using the actual HPC header.
    """
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'ERA5 NCVI source CSV missing: {path}')
    source=pd.read_csv(path)
    if source.empty:
        raise ValueError('ERA5 NCVI CSV is empty')
    def pick(override,candidates,kind):
        if override is not None:
            if override not in source.columns:
                raise ValueError(f'ERA5 NCVI configured {kind} field {override!r} absent; available={list(source.columns)!r}')
            return override
        options=[c for c in candidates if c in source.columns]
        if len(options)!=1:
            raise ValueError(f'ERA5 NCVI {kind} column ambiguous/missing: {options!r}; available={list(source.columns)!r}. Set ERA5_NCVI_{kind.upper()}_COL in Cell1 after inspecting the real CSV.')
        return options[0]
    date_col=pick(date_column,('date_bjt','bjt_date','date','Date'), 'date')
    # Only column names explicitly indicating the theta320 anomaly may be autodetected.
    value_col=pick(value_column,(
        'ncvi_anomaly_pvu','NCVI_anomaly_pvu','ncvi_theta320_anomaly_pvu','ncvi_theta320_anom_pvu',
        'ncvi_daily_anomaly_pvu','ncvi_anomaly','NCVI_anomaly',
        'ncvi_theta320_anomaly'), 'value')
    if date_col==value_col:
        raise ValueError('ERA5 NCVI date and value columns must differ')
    if expected_unit!='PVU':
        raise ValueError('ERA5 NCVI figure only accepts an input explicitly identified as PVU; no silent conversion')
    for col in ('units','unit'):
        if col in source.columns:
            declared=source[col].dropna().astype(str).str.strip().unique().tolist()
            if declared!=['PVU']:
                raise ValueError(f'ERA5 NCVI declared units are not unambiguously PVU: {declared}')
    dates=pd.to_datetime(source[date_col],errors='raise')
    if getattr(dates.dt,'tz',None) is not None:
        raise ValueError('ERA5 NCVI timestamps have a timezone; require explicit BJT calendar-day labels')
    if not (dates == dates.dt.normalize()).all():
        raise ValueError('ERA5 NCVI timestamps contain clock times; require explicit BJT calendar-day labels')
    values=pd.to_numeric(source[value_col],errors='raise')
    daily=pd.DataFrame({'date':dates,'value':values})
    daily=daily[daily.date.between('2023-06-14','2023-06-24')].sort_values('date').reset_index(drop=True)
    expected=pd.date_range('2023-06-14','2023-06-24',freq='D')
    if len(daily)!=11 or daily.date.duplicated().any() or not pd.DatetimeIndex(daily.date).equals(expected):
        raise ValueError('ERA5 NCVI must contain exactly one BJT daily value for each 14–24 June 2023')
    finite(daily.value,'ERA5 320 K NCVI anomaly (PVU)')
    return daily,{'path':str(path),'sha256':sha256(path),'date_column':date_col,
                  'value_column':value_col,'expected_unit':'PVU',
                  'units_evidence':'csv_units_field' if ('units' in source or 'unit' in source) else 'configuration_only_not_independently_verified',
                  'n_target_days':len(daily),'date_range':'2023-06-14/2023-06-24',
                  'time_label_assumption':'BJT calendar day; independent verification PENDING',
                  'spatial_reduction':'cos(latitude) weighted according to earlier method; not independently verified from this CSV',
                  'is_anomaly':'required by configured source identity; original climatology provenance PENDING',
                  'status':'PLOTTED_PROVENANCE_PENDING'}

original_run_hashes={str(p.relative_to(NCVI_RUN)):sha256(p) for p in NCVI_RUN.rglob('*') if p.is_file()}
nm=json.loads((NCVI_RUN/'ncvi_daily_export_manifest.json').read_text())
for name,digest in nm['outputs'].items():
 if Path(name).name!=name or original_run_hashes.get(name)!=digest:raise ValueError('NCVI run hash mismatch')
if nm['s2s_daily_source_qc']['status']!='PASS':raise ValueError('S2S physical source must PASS')
ncvi=pd.read_csv(NCVI_RUN/'table_ncvi_concurrent_daily_members.csv');ncvi['date']=pd.to_datetime(ncvi.date)
expected=pd.MultiIndex.from_product([INITS,pd.date_range('2023-06-14','2023-06-24'),range(51)],names=KEY)
if len(ncvi)!=1122 or ncvi.duplicated(KEY).any() or not pd.MultiIndex.from_frame(ncvi[KEY]).sort_values().equals(expected.sort_values()) or not ncvi.units.eq('PVU').all() or not ncvi.anomaly_used.eq(True).all():raise ValueError('NCVI coverage/units/anomaly invalid')
finite(ncvi.daily_ncvi_anomaly_pvu,'actual daily anomaly')
rq=pd.read_csv(NCVI_RUN/'table_ncvi_daily_stage_replay_qc.csv')
if len(rq)!=204 or rq.duplicated(['init_date','window','member']).any():raise ValueError('NCVI stage QC coverage invalid')
internal_error=max((rq.daily_equal_day_aggregate-rq.mlr_scalar_replay).abs().max(),(rq.daily_time_weighted_aggregate-rq.mlr_scalar_replay).abs().max())
if not np.isfinite(internal_error) or internal_error>1e-8:raise ValueError('Daily INTERNAL aggregation fails')
pieces=[]
for window in ['Stage-I','Total']:
 g=current_groups[current_groups.window==window]
 q=ncvi.merge(g[['init_date','member','group']],on=['init_date','member'],validate='many_to_one')
 if len(q)!=1122 or q.group.isna().any():raise ValueError('Group mapping failed')
 q['window']=window;q['factor']='NCVI_concurrent';q['value']=q.daily_ncvi_anomaly_pvu
 pieces.append(q[KEY+['window','group','factor','value']])
ng=group_daily(pd.concat(pieces,ignore_index=True))
all_physical=pd.concat([physical_group,ng],ignore_index=True)
# ERA5 physical references are independent existing observations, never S2S group means.
if ERA5_PHYSICAL is not None:
 ed=pd.read_csv(ERA5_PHYSICAL);hashes['era5_physical']={'path':str(ERA5_PHYSICAL),'sha256':sha256(ERA5_PHYSICAL)}
 erows=[]
 for init in INITS:
  for window,end,factors in [('Total','2023-06-24',FACTORS5),('Stage-I','2023-06-18',FACTORS11)]:
   for factor in factors:
    if factor=='NCVI_concurrent':continue
    z=ed[(ed.init==init)&(ed.grouping_stage==window)&(ed.factor==factor)].copy();z['date']=pd.to_datetime(z.date);z=z[z.date.between('2023-06-14',end)].sort_values('date')
    if len(z)!=len(pd.date_range('2023-06-14',end)) or z.date.duplicated().any():raise ValueError('ERA5 physical date coverage invalid')
    finite(z.era5,'ERA5 physical values')
    for r in z.itertuples():erows.append(dict(init_date=init,window=window,date=r.date,group='ERA5',factor=factor,value=r.era5))
 all_physical=pd.concat([all_physical,pd.DataFrame(erows)],ignore_index=True)

# Add ONE independent observation, replicated as the same ERA5 reference under
# each forecast initialization and under each Stage-I/Total window. S2S members
# are unchanged; this is NOT a grouped forecast product.
era5_ncvi_daily,era5_ncvi_audit=load_era5_ncvi_reference(
    ERA5_NCVI_CSV,ERA5_NCVI_DATE_COL,ERA5_NCVI_VALUE_COL,ERA5_NCVI_EXPECTED_UNIT)
hashes['era5_ncvi']={'path':str(ERA5_NCVI_CSV),'sha256':era5_ncvi_audit['sha256']}
era5_ncvi_rows=[]
for init in INITS:
 for window,stop in [('Total','2023-06-24'),('Stage-I','2023-06-18')]:
  for r in era5_ncvi_daily[era5_ncvi_daily.date<=pd.Timestamp(stop)].itertuples():
   era5_ncvi_rows.append(dict(init_date=init,window=window,date=r.date,
                              group='ERA5',factor='NCVI_concurrent',value=float(r.value)))
era5_ncvi_long=pd.DataFrame(era5_ncvi_rows)
all_physical=pd.concat([all_physical,era5_ncvi_long],ignore_index=True)
identity=['init_date','window','date','group','factor']
if all_physical.duplicated(identity).any():
 raise ValueError('ERA5 NCVI addition created duplicate init/window/date/group/factor rows')
for window in ('Total','Stage-I'):
 r0=era5_ncvi_long[(era5_ncvi_long.window==window)&(era5_ncvi_long.init_date==INITS[0])].sort_values('date')
 r1=era5_ncvi_long[(era5_ncvi_long.window==window)&(era5_ncvi_long.init_date==INITS[1])].sort_values('date')
 if not np.array_equal(r0.date.to_numpy(),r1.date.to_numpy()) or not np.array_equal(r0.value.to_numpy(),r1.value.to_numpy()):
  raise ValueError(f'ERA5 NCVI must be identical for both initializations: {window}')
fig5_table=all_physical[(all_physical.window=='Total')&all_physical.factor.isin(FACTORS5)].copy()
fig11_physical=all_physical[(all_physical.window=='Stage-I')&all_physical.factor.isin(FACTORS11)&(all_physical.date<=pd.Timestamp('2023-06-18'))].copy()
print('Cell 3 DIAGNOSTIC use ready; internal max',internal_error,'historical status preserved:',nm['status'],'ERA5 NCVI PLOTTED, PROVENANCE PENDING; column=',era5_ncvi_audit['value_column'])


In [ ]:
# Cell 4 — frozen saved-model partial fits; physical series identical by version.
capture(4)
partial=[];audit=[]
for init in INITS:
 for version in VERSIONS:
  model=parameters(data,init,version);relation=residual_relation(data,init)
  s=fig11_physical[(fig11_physical.init_date==init)&(fig11_physical.group!='ERA5')].pivot(index=['date','group'],columns='factor',values='value').reset_index()
  vals,heat=partial_fitted_tmax(s.sm_avg,s.SHF_Avg,model,relation)
  for j,r in enumerate(s.itertuples()):partial.append(dict(init_date=init,window='Stage-I',date=r.date,group=r.group,factor='partial_fitted_Tmax',value=vals[j],model_version=version,outside_training_window=r.date>pd.Timestamp('2023-06-17'),projected_heat=heat[j]))
  audit.append(dict(init_date=init,model_version=version,intercept=model['intercept'],fixed_residual_intercept=relation[0],fixed_residual_slope=relation[1],full_fit_error=model['full_fit_max_abs_difference']))
fig11_table=pd.concat([fig11_physical.assign(model_version=v) for v in VERSIONS]+[pd.DataFrame(partial)],ignore_index=True)
print('Cell 4 PASS; stored full-model replay and frozen Stage-I diagnostic; day18 out-of-window')

# Recompute member diagnostics separately from each stored complete model.
stage_members=pd.read_csv(STAGE);stage_members['window']=normalize_window(stage_members.stage)
scatter_rows=[]
for init in INITS:
 m=stage_members[(stage_members.init_date==init)&(stage_members.window=='Stage-I')].sort_values('member')
 if len(m)!=51 or not np.array_equal(m.member,np.arange(51)):raise ValueError('Scatter member coverage invalid')
 membership=current_groups[(current_groups.init_date==init)&(current_groups.window=='Stage-I')].set_index('member')
 for version in VERSIONS:
  model=parameters(data,init,version);relation=residual_relation(data,init)
  if not np.allclose(model['y'].to_numpy(),m.y_tmax.to_numpy(),rtol=0,atol=1e-8):raise ValueError('Scatter target/model mismatch')
  vals,heat=partial_fitted_tmax(m.sm_avg,m.SHF_Avg,model,relation)
  for j,r in enumerate(m.itertuples()):scatter_rows.append(dict(init_date=init,window='Stage-I',member=r.member,group=membership.loc[r.member,'group'],model_version=version,y_tmax=r.y_tmax,sm_avg=r.sm_avg,partial_fitted_Tmax=vals[j],projected_heat=heat[j]))
scatter_table=pd.DataFrame(scatter_rows)
paired=scatter_table.pivot(index=['init_date','member'],columns='model_version',values=['y_tmax','sm_avg'])
for variable in ['y_tmax','sm_avg']:
 if not np.array_equal(paired[variable]['raw'],paired[variable]['residualized']):raise ValueError('Raw SM/Tmax scatter changed across versions')
for variable in FACTORS11:
 a=fig11_table[(fig11_table.model_version=='raw')&(fig11_table.factor==variable)].sort_values(['init_date','date','group']).value.to_numpy()
 b=fig11_table[(fig11_table.model_version=='residualized')&(fig11_table.factor==variable)].sort_values(['init_date','date','group']).value.to_numpy()
 if not np.array_equal(a,b):raise ValueError('Physical timeseries changed across versions')
# Write dedicated plot inputs first; Cell5 reads these files, never old notebook objects.
if OUTPUT.exists():raise ValueError('New plot-table output directory required')
if original_run_hashes!={str(p.relative_to(NCVI_RUN)):sha256(p) for p in NCVI_RUN.rglob('*') if p.is_file()}:raise ValueError('Original NCVI run changed')
OUTPUT.mkdir(parents=True,exist_ok=False)
plot_tables={'fig5_total_timeseries_plot.csv':fig5_table}
for version in VERSIONS:
 plot_tables['fig11_stageI_timeseries_'+version+'_plot.csv']=fig11_table[fig11_table.model_version==version].copy()
 plot_tables['figS11_stageI_scatter_'+version+'.csv']=scatter_table[scatter_table.model_version==version].copy()
plot_hashes={}
for name,table in plot_tables.items():
 table.to_csv(OUTPUT/name,index=False);plot_hashes[name]=sha256(OUTPUT/name)
print('Cell 4 dedicated plot CSVs written; scatter and physical cross-version identity PASS')


In [ ]:
# Cell 5 — physical-size plotting/QC, no formal acceptance.
capture(5)
COLORS = {'ERA5': 'black', 'Hot17': 'firebrick', 'Middle17': '#1f77b4', 'Cold17': 'seagreen'}
LABELS = {'NCHN_tp': 'TP (mm day⁻¹)', 'z500_anom_NCHN': 'Z500 anomaly (gpm)',
          'sm_avg': 'SM (m³ m⁻³)', 'SHF_Avg': 'SHF (W m⁻²)',
          'NCVI_concurrent': 'NCVI anomaly (PVU)', 'TCC_Avg': 'TCC',
          'partial_fitted_Tmax': 'SM–SHF diagnostic (°C)'}
STYLE = {'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans'],
         'font.size': 9, 'axes.labelsize': 8, 'axes.titlesize': 9,
         'xtick.labelsize': 9, 'ytick.labelsize': 9,
         'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.bbox': None}
MARKERS = {'ERA5':'o','Hot17':'^','Middle17':'s','Cold17':'v'}
LINESTYLES = {'ERA5':'-','Hot17':'-','Middle17':'--','Cold17':':'}

def limits(values):
    a = np.asarray(values, float)
    a = a[np.isfinite(a)]
    if not len(a):
        raise ValueError('No finite values for panel range')
    lo, hi = float(a.min()), float(a.max())
    pad = max((hi-lo)*0.10, abs(lo)*0.01, 0.001)
    locator = MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10])
    ticks = locator.tick_values(lo-pad, hi+pad)
    return (float(ticks[0]), float(ticks[-1])), ticks

def legend(fig, reference=True):
    names = ('ERA5', *GROUPS) if reference else GROUPS
    handles = [Line2D([0], [0], color=COLORS[g], marker=MARKERS[g], linestyle=LINESTYLES[g], lw=1.4, label=g) for g in names]
    fig.legend(handles=handles, loc='lower center', ncol=len(names), frameon=False,
               bbox_to_anchor=(0.5, 0.005), fontsize=8)

def style(ax):
    ax.grid(True, linestyle=':', alpha=0.2, linewidth=0.4)
    ax.tick_params(width=0.7, length=3)
    for spine in ax.spines.values():
        spine.set_color('0.5'); spine.set_linewidth(0.6)

def timeseries_figure(df, factors, window, version=None):
    rows = len(factors)
    fig, axes = plt.subplots(rows, 2, figsize=(6.5, 8.6 if rows == 5 else 8.9), dpi=600)
    data = df if version is None else df[df.model_version == version]
    end = '2023-06-24' if window == 'Total' else '2023-06-18'
    # The labelled BJT calendar day is the exact plotted time coordinate.
    # Never introduce the old +12 h display offset: markers and daily ticks coincide.
    dates = pd.date_range('2023-06-14', end, freq='D')
    # Pool both inits and both parameterizations for identical row scales.
    # A missing ERA5 NCVI line must never be silently dropped.
    for row, factor in enumerate(factors):
        ylim, ticks = limits(df[df.factor == factor].value)
        if factor == 'TCC_Avg':
            # TCC is a physical fraction. Preserve values (never rescale or clip).
            # Fixed bounds avoid MaxNLocator's artificial 1.25 top tick.
            tcc_values = np.asarray(df.loc[df.factor == factor, 'value'], dtype=float)
            if (not np.isfinite(tcc_values).all() or
                    np.any((tcc_values < 0.0) | (tcc_values > 1.0))):
                raise ValueError('TCC values are outside the physical fraction [0, 1]; verify source units')
            ylim, ticks = (0.0, 1.0), np.array([0.0, 0.25, 0.50, 0.75, 1.0])
        for col, init in enumerate(INITS):
            ax = axes[row, col]
            s = data[(data.init_date == init) & (data.factor == factor)].copy()
            s['date'] = pd.to_datetime(s.date)
            for group in ('ERA5', *GROUPS):
                g = s[s.group == group].sort_values('date')
                if g.empty and group == 'ERA5':
                    if factor == 'NCVI_concurrent':
                        raise ValueError(f'Mandatory ERA5 NCVI reference missing: {init}/{window}')
                    # ERA5 for other physical variables remains optional.
                    continue
                if len(g) != len(dates) or not np.array_equal(pd.DatetimeIndex(g.date).values, dates.values):
                    raise ValueError(f'Incomplete or misaligned daily plot series: {init}/{factor}/{group}')
                if not np.isfinite(g.value.to_numpy(dtype=float)).all():
                    raise ValueError(f'Nonfinite plot series: {init}/{factor}/{group}')
                ax.plot(g.date, g.value, color=COLORS[group], marker=MARKERS[group],
                        linestyle=LINESTYLES[group], markersize=4.2, markeredgewidth=0.5,
                        markeredgecolor=COLORS[group], lw=1.3)
            ax.set_ylim(ylim); ax.set_yticks(ticks)
            # Half-day padding keeps endpoint markers fully visible, without shifting data.
            ax.set_xlim(dates[0]-pd.Timedelta(hours=12), dates[-1]+pd.Timedelta(hours=12))
            ax.set_xticks(dates)  # every daily node, including all 11 days of Fig.5
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
            if row == rows-1:
                # Eleven '06-14'...'06-24' labels cannot fit at 45 degrees in
                # two 165-mm-wide columns. 70 degrees keeps ALL dates readable.
                angle = 70 if window == 'Total' else 0
                ax.tick_params(axis='x', labelbottom=True, labelsize=8 if window == 'Total' else 9)
                plt.setp(ax.get_xticklabels(), rotation=angle,
                         ha='right' if angle else 'center', rotation_mode='anchor')
            else:
                ax.tick_params(axis='x', labelbottom=False)
            if window == 'Stage-I':
                ax.axvline(pd.Timestamp('2023-06-18 00:00'), color='0.4', linestyle='--', lw=0.8)
            if window == 'Stage-I':
                ax.axvspan(pd.Timestamp('2023-06-18'), pd.Timestamp('2023-06-19'), color='0.9', alpha=0.5, zorder=-1)
            ax.set_title(f'({chr(97+row*2+col)})', loc='left')
            ax.set_ylabel(LABELS[factor] if col == 0 else '')
            style(ax)
    for col, init in enumerate(INITS):
        pos=axes[0,col].get_position()
        fig.text(0.345 if col==0 else 0.79,0.965,pd.Timestamp(init).strftime('%d June %Y'),ha='center',fontsize=10,fontweight='semibold')
    legend(fig, reference=bool((data.group=='ERA5').any()))
    fig.subplots_adjust(left=0.15, right=0.98, top=0.91,
                        bottom=0.115 if window == 'Total' else 0.09,
                        wspace=0.24, hspace=0.43)
    return fig, axes.ravel().tolist()

def pdf_page_qc(page, expected_height_inches):
    """Actual PDF measurements, with separate superscript/subscript allowance.

    Ordinary spans require >=8pt; flagged superscripts/subscripts >=5.5pt.
    Unflagged unicode superscripts in ordinary text remain ordinary spans.
    These are explicit readability criteria, not a claim AGU mandates these minima.
    """
    spans=[span for block in page.get_text('dict')['blocks'] if 'lines' in block for line in block['lines'] for span in line['spans'] if span['text'].strip()]
    body=[span for span in spans if not (span.get('flags',0)&1)]
    scripts=[span for span in spans if span.get('flags',0)&1]
    reasons=[]
    width_mm=page.rect.width/72*25.4;height_mm=page.rect.height/72*25.4
    if width_mm>170+.01 or height_mm>228+.01:reasons.append('AGU physical size limit exceeded')
    if abs(page.rect.width-6.5*72)>.1 or abs(page.rect.height-expected_height_inches*72)>.1:reasons.append('Export canvas differs from intended size')
    if not body:reasons.append('No editable ordinary text')
    if any(span['size']<7.99 for span in body):reasons.append('Ordinary label below 8pt')
    if any(span['size']<5.49 for span in scripts):reasons.append('Superscript/subscript below 5.5pt')
    if any(span['bbox'][0]<-.1 or span['bbox'][1]<-.1 or span['bbox'][2]>page.rect.width+.1 or span['bbox'][3]>page.rect.height+.1 for span in spans):reasons.append('Actual PDF text clipped')
    return dict(status='FAIL' if reasons else 'PASS',reasons='; '.join(reasons),width_mm=width_mm,height_mm=height_mm,
                min_body_pt=min((span['size'] for span in body),default=None),min_script_pt=min((span['size'] for span in scripts),default=None),
                body_spans=len(body),script_spans=len(scripts),editable_text_spans=len(spans),body_minimum_pt=8,script_minimum_pt=5.5)

def figure_qc(fig, axes):
    if fig.get_figwidth()*25.4>170 or fig.get_figheight()*25.4>228:raise ValueError('AGU physical dimensions exceeded')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    sizes = np.array([[a.get_position().width, a.get_position().height] for a in axes])
    if not np.allclose(sizes, sizes[0], rtol=0, atol=1e-10):
        raise ValueError('Unequal process-panel axes boxes')
    daily_ticks_aligned = None
    total_dates_all_labelled = None
    if len(axes) in (10, 12):
        window_end = '2023-06-24' if len(axes) == 10 else '2023-06-18'
        expected_dates = pd.date_range('2023-06-14', window_end, freq='D')
        expected_x = mdates.date2num(expected_dates.to_pydatetime())
        for ax in axes:
            if not np.allclose(ax.get_xticks(), expected_x, atol=1e-9, rtol=0):
                raise ValueError('Daily tick locations do not match calendar-day nodes')
            member_lines = [line for line in ax.lines if line.get_marker() in set(MARKERS.values())]
            if len(member_lines) not in (3, 4):
                raise ValueError('Expected three S2S group curves and optional ERA5')
            for line in member_lines:
                line_dates = pd.DatetimeIndex(pd.to_datetime(line.get_xdata(orig=True)))
                if not np.array_equal(line_dates.values, expected_dates.values):
                    raise ValueError('Time-series marker lies between daily tick nodes')
        daily_ticks_aligned = True
        if len(axes) == 10:
            required_labels = [d.strftime('%m-%d') for d in expected_dates]
            for ax in axes[-2:]:
                labels = ax.get_xticklabels()
                if [t.get_text() for t in labels] != required_labels:
                    raise ValueError('Fig.5 does not label every day, 06-14 through 06-24')
                if not all(65 <= t.get_rotation() <= 75 for t in labels):
                    raise ValueError('Fig.5 daily tick labels require angled display')
                spans = [t.get_window_extent(renderer) for t in labels]
                if any(a.overlaps(b) for a, b in zip(spans[:-1], spans[1:])):
                    raise ValueError('Fig.5 angled daily labels overlap')
            total_dates_all_labelled = True
    if len(axes) == 4:
        # Explicitly validate the user-specified asymmetric annotation layout.
        for i, ax in enumerate(axes):
            boxes = [t for t in ax.texts if 'n = 51' in t.get_text()]
            if len(boxes) != 1:
                raise ValueError('Fig.S11 must show exactly one statistics box per panel')
            expected_ha = 'right' if i < 2 else 'left'
            if boxes[0].get_ha() != expected_ha:
                raise ValueError(f'Fig.S11 statistics box misplaced in panel {i}')
    texts = list(fig.texts)
    for ax in axes:
        texts += [ax._left_title, ax.xaxis.label, ax.yaxis.label] + list(ax.texts) + ax.get_xticklabels() + ax.get_yticklabels()
    boxes = [(t.get_text(), t.get_window_extent(renderer)) for t in texts if t.get_visible() and t.get_text()]
    for label, b in boxes:
        if b.x0 < 0 or b.y0 < 0 or b.x1 > fig.bbox.width or b.y1 > fig.bbox.height:
            raise ValueError(f'Clipped text in process figure: {label}')
    for i, (label, a) in enumerate(boxes):
        for other, b in boxes[i+1:]:
            if min(a.x1,b.x1)-max(a.x0,b.x0)>1 and min(a.y1,b.y1)-max(a.y0,b.y0)>1:
                raise ValueError(f'Overlapping process labels: {label}; {other}')
    return {'equal_boxes': True, 'text_extent_qc': 'PASS',
            'daily_marker_tick_alignment': daily_ticks_aligned,
            'fig5_all_11_days_labelled': total_dates_all_labelled,
            'width_inches': fig.get_figwidth(),
            'height_inches': fig.get_figheight(), 'axes_count': len(axes)}

def scatter_statistics(df, version, init, factor):
    members=df[(df.init_date==init)&(df.model_version==version)].sort_values('member')
    if len(members)!=51 or not np.array_equal(members.member.to_numpy(),np.arange(51)):
        raise ValueError('Descriptive scatter statistics require 51 original members')
    x=members[factor].to_numpy(float);y=members.y_tmax.to_numpy(float)
    if not np.isfinite(x).all() or not np.isfinite(y).all() or np.std(x)==0 or np.std(y)==0:
        raise ValueError('Pearson/trend undefined for nonfinite or constant data')
    dx=x-x.mean();dy=y-y.mean()
    r=float(np.dot(dx,dy)/np.sqrt(np.dot(dx,dx)*np.dot(dy,dy)))
    slope=float(np.dot(dx,dy)/np.dot(dx,dx));intercept=float(y.mean()-slope*x.mean())
    return dict(init_date=init,model_version=version,factor=factor,n=51,pearson_r=r,
                descriptive_R2=r*r,trend_slope=slope,trend_intercept=intercept,
                R2_definition='Squared pairwise Pearson correlation; not complete MLR R2')

def scatter_figure(df, version):
    fig, axes = plt.subplots(2, 2, figsize=(6.5, 5.9), dpi=600)
    for row, factor in enumerate(('sm_avg', 'partial_fitted_Tmax')):
        for col, init in enumerate(INITS):
            ax = axes[row, col]
            all_versions = df[df.init_date == init]
            g = all_versions[all_versions.model_version == version]
            for group in GROUPS:
                s = g[g.group == group]
                ax.scatter(s[factor], s.y_tmax, s=23, marker=MARKERS[group], color=COLORS[group], alpha=0.7,
                           edgecolor='white', linewidth=0.3)
            # Paired versions share limits; the SM panel has identical input points.
            xlim, xticks = limits(df[factor])
            ylim, yticks = limits(df.y_tmax)
            stats=scatter_statistics(df,version,init,factor)
            xx=np.array(xlim)
            ax.plot(xx,stats['trend_intercept']+stats['trend_slope']*xx,color='0.3',lw=1.0,ls='--',zorder=1)
            # Tmax–SM (top row): move statistics to the right.
            # Tmax–SM/SHF partial fit (bottom row): retain LEFT-side statistics.
            # This is Pearson r^2, NOT the six-predictor MLR R^2.
            stat_x, stat_ha = ((0.975, 'right') if factor == 'sm_avg'
                               else (0.025, 'left'))
            ax.text(stat_x,0.965,
                    f"r = {stats['pearson_r']:.2f}\nR² = r² = {stats['descriptive_R2']:.2f}\nn = 51",
                    transform=ax.transAxes,ha=stat_ha,va='top',fontsize=8,
                    bbox=dict(facecolor='white',edgecolor='none',alpha=0.88,pad=2.0))
            ax.set_xlim(xlim); ax.set_xticks(xticks)
            ax.set_ylim(ylim); ax.set_yticks(yticks)
            ax.set_xlabel(LABELS[factor]); ax.set_ylabel('Stage-I Tmax (°C)')
            ax.set_title(f'({chr(97+row*2+col)})', loc='left')
            style(ax)
    for col,init in enumerate(INITS):
        fig.text(0.33 if col==0 else 0.79,0.975,pd.Timestamp(init).strftime('%d June %Y'),ha='center',fontsize=10,fontweight='semibold')
    legend(fig, reference=False)
    fig.subplots_adjust(left=0.14, right=0.955, top=0.94, bottom=0.14, wspace=0.40, hspace=0.50)
    return fig, axes.ravel().tolist()

# Read and verify dedicated CSVs before drawing.
for name,expected_hash in plot_hashes.items():
 if sha256(OUTPUT/name)!=expected_hash:raise ValueError('Plot CSV changed')
fig5_plot=pd.read_csv(OUTPUT/'fig5_total_timeseries_plot.csv')
fig11_plot=pd.concat([pd.read_csv(OUTPUT/('fig11_stageI_timeseries_'+v+'_plot.csv')) for v in VERSIONS],ignore_index=True)
scatter_plot=pd.concat([pd.read_csv(OUTPUT/('figS11_stageI_scatter_'+v+'.csv')) for v in VERSIONS],ignore_index=True)
built=[];layout_qc=[]
with plt.rc_context(STYLE):
 fig,axes=timeseries_figure(fig5_plot,FACTORS5,'Total');built.append(('Fig5_Total_DIAGNOSTIC',fig,axes))
 for version in VERSIONS:
  fig,axes=timeseries_figure(fig11_plot,('partial_fitted_Tmax',*FACTORS11),'Stage-I',version)
  built.append(('Fig11_StageI_'+version+'_DIAGNOSTIC',fig,axes))
  fig,axes=scatter_figure(scatter_plot,version);built.append(('FigS11_StageI_'+version+'_DIAGNOSTIC',fig,axes))
 for name,fig,axes in built:
  layout_qc.append(dict(filename=name,**figure_qc(fig,axes)))
  display(fig)
scatter_stats=pd.DataFrame([scatter_statistics(scatter_plot,v,init,factor) for v in VERSIONS for init in INITS for factor in ['sm_avg','partial_fitted_Tmax']])
print('Cell 5 layout QC PASS; all figures DIAGNOSTIC, 6.5 inch / PNG600 dpi / vector PDF')


In [ ]:
# Cell 6 — independent export; no history/status modification.
capture(6)
if set(SNAPSHOTS)!={1,2,3,4,5,6}:raise RuntimeError('Run all cells in order')
if not OUTPUT.is_dir():raise ValueError('Run Cell4 plot-table export first')
for name,expected_hash in plot_hashes.items():
 if sha256(OUTPUT/name)!=expected_hash:raise ValueError('Plot CSV changed')
if original_run_hashes!={str(p.relative_to(NCVI_RUN)):sha256(p) for p in NCVI_RUN.rglob('*') if p.is_file()}:raise ValueError('Original run changed')
for key,item in hashes.items():
 if sha256(item['path'])!=item['sha256']:raise ValueError('Input changed')
tables={'v12_daily_stage_qc.csv':physical_qc,'group_membership_audit.csv':current_groups,'partial_fitted_audit.csv':pd.DataFrame(audit),
'layout_qc.csv':pd.DataFrame(layout_qc),'scatter_descriptive_statistics.csv':scatter_stats}
for name,table in tables.items():
 if (OUTPUT/name).exists():raise ValueError('Never overwrite audit output')
 table.to_csv(OUTPUT/name,index=False)
for name,fig,axes in built:
 if any((OUTPUT/(name+suffix)).exists() for suffix in ['.png','.pdf','.svg']):raise ValueError('Never overwrite figure')
 fig.savefig(OUTPUT/(name+'.png'),dpi=600)
 fig.savefig(OUTPUT/(name+'.pdf'))
 fig.savefig(OUTPUT/(name+'.svg'))
if (OUTPUT/'executed_cells.json').exists() or (OUTPUT/'consistency_manifest.json').exists():raise ValueError('Never overwrite provenance')
snapshot=json.dumps(SNAPSHOTS,indent=2,ensure_ascii=False)
(OUTPUT/'executed_cells.json').write_text(snapshot,encoding='utf-8')
pdf_checks=[]
try:
 import fitz
 for name,fig,axes in built:
  with fitz.open(OUTPUT/(name+'.pdf')) as doc:
   pdf_checks.append(dict(filename=name,**pdf_page_qc(doc[0],fig.get_figheight())))
except ImportError:
 pdf_checks=[dict(status='NOT_RUN',reason='PyMuPDF unavailable; renderer QA passed, actual PDF audit pending')]
pd.DataFrame(pdf_checks).to_csv(OUTPUT/'pdf_font_qc.csv',index=False)
report={'status' :'PRELIMINARY_DIAGNOSTIC','formal_process_allowed':False,'notebook_source_commit':'7f72d69','notebook_source_commit_role':'PR plot-layout revision immediately before provenance refresh; executed_code_sha256 identifies actual HPC code','historical_reference_baseline':'90b4982','reference_commit':'90b4982','reference_commit_role':'reviewed baseline, not HPC HEAD','executed_code_sha256':hashlib.sha256(snapshot.encode('utf-8')).hexdigest(),'execution_git_head':None,
'inputs':hashes,'original_ncvi_run_sha256':original_run_hashes,'original_ncvi_manifest':nm,
'daily_internal_aggregation_qc':{'status':'PASS','max_difference':float(internal_error)},
'historical_mlr_strict_qc_preserved':nm['s2s_stage_replay_qc'],'era5_ncvi':'PLOTTED_PROVENANCE_PENDING','era5_ncvi_source_qc':era5_ncvi_audit,
'pdf_font_qc':pdf_checks,'scatter_R2_definition':'pairwise Pearson r squared, not complete MLR R2',
'physical_units':{'NCHN_tp':'mm per source daily interval','sm_avg':'m3 m-3','SHF_Avg':'W m-2','z500_anom_NCHN':'gpm','TCC_Avg':'fraction','NCVI_concurrent':'PVU'},
'methods_notes':['v12 Tmax arithmetic grid mean; legacy area-weighted definition differs','TCC source interval 08:00–08:00 BJT; source date labels retained','day18 partial fit is frozen Stage-I out-of-window projection','raw/residualized diagnostic differences are parameterization sensitivity'],
'outputs':{p.name:sha256(p) for p in OUTPUT.iterdir() if p.is_file()}}
(OUTPUT/'consistency_manifest.json').write_text(json.dumps(report,indent=2,default=str))
print('Cell 6 DIAGNOSTIC exports complete:',OUTPUT,'formal_process_allowed=False; original run unchanged')

if any(item['status']=='FAIL' for item in pdf_checks):raise ValueError('Actual PDF QC FAIL; see saved pdf_font_qc.csv; no acceptance claimed')
print('Actual PDF QC:',pdf_checks)
